<a href="https://colab.research.google.com/github/fidlarsyn/DL-Practical-Statistics-for-Data-Scientists-O-Reilly/blob/main/Bab_4_Regression_and_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ringkasan**
### Regresi adalah metode statistik utama untuk memodelkan dan mengkuantifikasi hubungan antara satu atau beberapa variabel prediktor (*independent variables/features*) dengan variabel respon numerik (*dependent variable/outcome*). Bab ini mengulas regresi linier sederhana, regresi berganda, inferensi versus prediksi, pengolahan variabel kategorikal, diagnostik sisaan (*residuals*), hingga regresi non-linier (*splines* &amp; GAM).

# **1. Simple Linear Regression**
### Regresi linier sederhana memodelkan hubungan antara satu variabel prediktor ($X$) dan satu variabel respon ($Y$) dalam bentuk garis lurus. Perbedaannya dengan korelasi adalah jika korelasi hanya mengukur kekuatan asosiasi simetris antara dua variabel, regresi mengkuantifikasi hubungan kausal/prediktif spesifik di mana perubahan pada $X$ memicu perubahan estimasi pada $Y$.

**Code reproduction: R:**

In [ ]:
# Memodelkan Peak Expiratory Flow Rate (PEFR) terhadap durasi pendedahan debu kapas (Exposure)
model <- lm(PEFR ~ Exposure, data=lung)
summary(model)

# **2. The Regression Equation**
### Persamaan regresi linier dituliskan sebagai $Y = b_0 + b_1 X + e$. Nilai $b_0$ (intercept) melambangkan perkiraan nilai $Y$ saat $X = 0$. Nilai $b_1$ (slope / regression coefficient) menunjukkan besarnya perubahan estimasi $Y$ untuk setiap peningkatan satu unit pada $X$.

# **3. Fitted Values and Residuals**
### Fitted values ($\hat{Y}_i$) adalah nilai prediksi yang dihasilkan oleh garis regresi. Residuals ($e_i = Y_i - \hat{Y}_i$) adalah selisih atau eror antara nilai pengamatan asli dengan nilai prediksi.

**Code Reproduction: R:**

In [ ]:
fitted <- predict(model)
resid <- residuals(model)$0

# **4. Least Squares**
### Garis regresi ditentukan menggunakan metode kuadrat terkecil (Ordinary Least Squares / OLS). OLS mencari nilai koefisien $b_0$ dan $b_1$ yang meminimalkan Jumlah Kuadrat Sisaan (Residual Sum of Squares / RSS): $$ \text{RSS} = \sum_{i=1}^n (Y_i - \hat{Y}_i)^2 $$

# **5. Prediction Versus Explanation (Profiling)**
* **Explanation:** Berfokus pada analisis historis untuk memahami hubungan variabel (perhatian utama pada estimasi koefisien $b_1$).
* **Prediction:** Berfokus pada pemodelan data science untuk memprediksi nilai $Y$ baru di masa depan secara akurat (perhatian utama pada nilai prediksi $\hat{Y}$ dan out-of-sample error).

# **6. Multiple Linear Regression**
### Memperluas regresi linier untuk menyertakan banyak prediktor: $Y = b_0 + b_1 X_1 + b_2 X_2 + \dots + b_p X_p + e$. Setiap koefisien $b_j$ mewakili perubahan estimasi $Y$ per unit peningkatan $X_j$, dengan asumsi seluruh prediktor lainnya bernilai konstan (ceteris paribus).

# **7. Example: King County Housing Data**
### Menggunakan dataset harga rumah King County (Seattle) untuk memprediksi harga jual terkoreksi (`AdjSalePrice`) berdasarkan luas rumah (`SqFtTotLiving`), luas tanah (`SqFtLot`), jumlah kamar mandi (`Bathrooms`), jumlah kamar tidur (`Bedrooms`), dan kelas bangunan (`BldgGrade`).

**Code Reproduction: R:**

In [ ]:
house_lm <- lm(AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms + BldgGrade,
               data=house, na.action=na.omit)
summary(house_lm)

# **8. Assessing the Model**
### Metrik evaluasi utama dalam data science adalah Root Mean Squared Error (RMSE) yang mengukur akurasi keseluruhan kesalahan prediksi: $$ \text{RMSE} = \sqrt{\frac{\sum (y_i - \hat{y}_i)^2}{n}} $$ Residual Standard Error (RSE) menyesuaikan penyeimbang dengan derajat kebebasan $n - p - 1$. Koefisien Determinasi ($R^2$) mengukur proporsi variansi data yang dapat dijelaskan oleh model (bernilai 0 hingga 1).

**code Reproduction: R:**

In [ ]:
summary(house_lm)# Menampilkan RSE, Multiple R-squared, Adjusted R-squared, F-statistic$0

# **9. Cross-Validation**
### K-fold cross-validation membagi data menjadi $k$ bagian (folds). Model dilatih pada $k-1$ bagian dan diuji pada 1 bagian tahanan (holdout sample) secara bergantian. Evaluasi out-of-sample ini mencegah generalisasi yang buruk akibat kebetulan pada satu sampel uji.

# **10. Model Selection and Stepwise Regression**
### Menerapkan prinsip Occam’s razor (memilih model paling sederhana dengan kinerja setara). Penambahan variabel prediktor selalu menurunkan training RMSE tetapi berisiko memicu overfitting. Metode otomatis seperti Stepwise Regression menggunakan kriteria Akaike Information Criterion (AIC) atau Adjusted $R^2$ untuk menambah (forward selection) atau membuang (backward elimination) variabel secara bertahap.

**Code Reproduction: R:**

In [ ]:
library(MASS)
house_full <- lm(AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms +
                 Bedrooms + BldgGrade + PropertyType + NbrLivingUnits +
                 SqFtFinBasement + YrBuilt + YrRenovated + NewConstruction,
               data=house, na.action=na.omit)

step <- stepAIC(house_full, direction="both")
step$

# **11. Weighted Regression**
### Digunakan ketika observasi memiliki tingkat presisi/variansi yang berbeda (*inverse-variance weighting*) atau ketika satu baris mewakili agregasi beberapa kasus. Kasus nyata: data penjualan rumah lama diberi bobot lebih kecil dibanding penjualan rumah terbaru.

**Code Reproduction: R:**

In [ ]:
house\$Year <- year(house\$DocumentDate)
house\$Weight <- house\$Year - 2005

house_wt <- lm(AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms + BldgGrade,
               data=house, weight=Weight)$0

# **12. Prediction Using Regression**
### Tujuan utama regresi dalam *data science* adalah memprediksi observasi baru yang belum pernah dilihat sebelumnya.

# **13. The Dangers of Extrapolation**
### Model regresi tidak boleh digunakan untuk melakukan ekstrapolasi (memprediksi di luar rentang nilai data latih). Misalnya, memprediksi harga sebidang tanah kosong menggunakan model yang hanya dilatih pada data rumah berbangunan akan menghasilkan estimasi harga negatif yang tidak masuk akal.

# **14. Confidence and Prediction Intervals**

  * **Confidence Interval**: Mengukur ketidakpastian seputar estimasi nilai rata-rata garis regresi/koefisien.
  * **Prediction Interval**: Mengukur ketidakpastian untuk **satu nilai observasi individu** baru. Rentangnya jauh lebih lebar karena memperhitungkan variasi alami individual (*residual error*).

**Code Reproduction: R:**

In [ ]:
# Menghitung Prediction Interval untuk sampel baru
newhouse <- house[1, predictors]
predict(house_lm, newdata=newhouse, interval="prediction")

# **15. Factor Variables in Regression**
### Variabel faktor (*categorical variables*) memiliki sekumpulan nilai diskrit terbatas. Karena persamaan regresi membutuhkan input numerik, variabel faktor harus dikodekan ulang menjadi variabel biner / indikator (*dummy variables*).

# **16. Dummy Variables Representation**
### Untuk faktor dengan $P$ tingkat kategori, regresi linier hanya menggunakan $P - 1$ kolom dummy variable. Satu tingkat dikorbankan sebagai tingkat acuan (reference level) untuk menghindari dummy variable trap atau multicollinearity. Dalam machine learning non-regresi (seperti tree), pengkodean seluruh $P$ kolom disebut One-Hot Encoding.

**Code Repodruction: R:**

In [ ]:
# Membuat matriks dummy P-1 secara otomatis
lm(AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms + BldgGrade + PropertyType,
   data=house)$0

# **17. Factor Variables with Many Levels**
### Faktor dengan kategori sangat banyak (seperti kode pos / *zip code*) dapat menghasilkan puluhan kolom *dummy* berlebih. Solusinya adalah mengonsolidasikan kelompok kode pos berdasarkan karakteristik geografis awal atau berdasarkan median sisaan (*median residuals*) dari model awal ke dalam beberapa kelompok kuantil (`ZipGroup`).

**Code Repodruction: R:**

In [ ]:
zip_groups <- house %>%
  mutate(resid = residuals(house_lm)) %>%
  group_by(ZipCode) %>%
  summarize(med_resid = median(resid), cnt = n()) %>%
  arrange(med_resid) %>%
  mutate(cum_cnt = cumsum(cnt), ZipGroup = ntile(cum_cnt, 5))

house <- house %>% left_join(select(zip_groups, ZipCode, ZipGroup), by='ZipCode')

# **18. Ordered Factor Variables**

### Variabel kategorikal berurut (*ordinal*) seperti kelas kualitas bangunan (`BldgGrade`: 1 = Substandard, 13 = Mansion) sering kali dapat dimasukkan secara langsung sebagai variabel numerik tunggal untuk mempertahankan informasi urutannya.

# **19. Interpreting the Regression Equation**
### Menginterpretasikan tanda (+/-) dan besarnya koefisien regresi memerlukan kehati-hatian, terutama ketika antar prediktor saling berhubungan.

# **20. Correlated Predictors**
### Jika variabel prediktor saling berkorelasi (misal: jumlah kamar tidur dan luas total rumah), tanda koefisien regresi bisa kontra-intuitif (misal: koefisien kamar tidur bernilai negatif). Hal ini terjadi karena kamar tidur bertindak sebagai koreksi terhadap luas rumah untuk ukuran rumah yang sama.

**Code Repodruction: R:**

In [ ]:
# Menghapus variabel luas total memperlihatkan koefisien kamar tidur berubah menjadi positif
update(step_lm, . ~ . - SqFtTotLiving - SqFtFinBasement - Bathrooms)

# **21. Multicollinearity**

### *Multicollinearity* adalah kondisi ekstrem di mana prediktor memiliki korelasi sempurna atau mendekati sempurna satu sama lain. Menyebabkan matriks regresi tidak stabil atau gagal dihitung. Solusinya adalah membuang variabel yang redundan (misalnya memastikan pengkodean *dummy* membuang 1 kolom acuan).

# **22. Confounding Variables**

### Masalah *omission* (kelalaian): variabel penting yang relevan tidak dimasukkan ke dalam persamaan regresi (misal: variabel lokasi rumah). Mengabaikan variabel pengganggu (*confounding variable*) menyebabkan koefisien variabel lain menjadi bias atau semu.

**Code Reproduction: R:**

In [ ]:
# Memasukkan ZipGroup memperbaiki tanda koefisien kamar mandi dan luas tanah
lm(AdjSalePrice ~ SqFtTotLiving + SqFtLot + Bathrooms + Bedrooms + BldgGrade + PropertyType + ZipGroup,
   data=house)

# **23. Interactions and Main Effects**
### Main effects mengasumsikan hubungan prediktor dan respon bersifat independen. Interaction term (ditulis $X_1 \times X_2$) dimasukkan jika pengaruh satu variabel bergantung pada tingkatan variabel lain (misal: penambahan luas rumah di lokasi mewah akan menaikkan harga jauh lebih tinggi dibanding di lokasi murah).

**Code Repodruction: R:**

In [ ]:
# Operator * menambahkan main effects sekaligus interaction terms
lm(AdjSalePrice ~ SqFtTotLiving * ZipGroup + SqFtLot + Bathrooms + Bedrooms + BldgGrade,
   data=house)

# **24. Regression Diagnostics**
### Langkah-langkah evaluasi sisaan (*residuals*) untuk mengecek kecocokan asumsi model. *Standardized Residuals* dihitung dengan membagi sisaan dengan standar eror sisaan.

# **25. Outliers**
### *Outlier* dalam regresi adalah observasi yang memiliki sisaan (*residual*) sangat besar (berjarak jauh dari garis prediksi). Bermanfaat untuk mendeteksi anomali data (seperti kesalahan entri data atau penjualan parsial kepemilikan).

**Code Repodruction: R: **

In [ ]:
sresid <- rstandard(lm_98105)
idx <- order(sresid)
sresid[idx[1]]  # Mengidentifikasi sisaan negatif terbesar$0

# **26. Influential Values**
### Observasi yang keberadaan atau ketidakberadaannya mengubah persamaan regresi secara drastis disebut memiliki *leverage* tinggi. Diukur menggunakan kombinasi *Hat-values* (leverage) dan *Cook’s Distance*.

**Code Repodruction: R: **

In [ ]:
# Bubble plot nilai Cook's distance
cooks_D <- cooks.distance(lm_98105)
hat_values <- hatvalues(lm_98105)
plot(hat_values, rstandard(lm_98105), cex=10*sqrt(cooks_D))$0

# **27. Heteroskedasticity, Non-Normality, and Correlated Errors**

  * **Heteroskedasticity**: Kondisi di mana variansi sisaan tidak konstan di seluruh rentang nilai prediksi (misal: eror prediksi rumah mahal jauh lebih besar dibanding rumah murah). Menunjukkan adanya prediktor yang belum terakomodasi.
  * **Non-Normality**: Sisaan yang ber-ekor tebal (*long-tailed*).
  * **Correlated Errors**: Terdapat autokorelasi pada sisaan (diuji via statistik *Durbin-Watson*), umum terjadi pada data deret waktu (*time series*).

**Code Reproduction: R:**

In [ ]:
# Plot absolut sisaan vs nilai prediksi
df <- data.frame(resid = residuals(lm_98105), pred = predict(lm_98105))
ggplot(df, aes(pred, abs(resid))) + geom_point() + geom_smooth()$0

# **28. Partial Residual Plots and Nonlinearity**
### Partial residual plot mengisolasi hubungan antara satu variabel prediktor tunggal dengan respon setelah memperhitungkan efek seluruh prediktor lainnya. Berguna untuk mendeteksi secara visual apakah hubungan bersifat non-linier. $$ \text{Partial Residual} = \text{Residual} + b_i X_i $$

**Code Reproduction: R:**

In [ ]:
terms <- predict(lm_98105, type='terms')
partial_resid <- resid(lm_98105) + terms
# Plot partial residual SqFtTotLiving$0

# **29. Polynomial and Spline Regression**
### Ekstensi regresi linier untuk menangkap pola melengkung / non-linier.

# **30. Polynomial**
### Menambahkan suku polinomial (seperti kuadrat $X^2$ atau kubik $X^3$) ke dalam persamaan regresi.

**Code Reproduction: R:**

In [ ]:
lm(AdjSalePrice ~ poly(SqFtTotLiving, 2) + SqFtLot + BldgGrade + Bathrooms + Bedrooms,
   data=house_98105)

#**31. Splines**
### *Spline* membagi rentang variabel prediktor menjadi beberapa segmen polinomial mulus yang terhubung di titik-titik simpul (*knots*). *B-spline (Basis Spline)* memberikan fleksibilitas pemodelan non-linier yang jauh lebih stabil dibanding polinomial derajat tinggi.

**Code Reproduction: R:**

In [ ]:
library(splines)
knots <- quantile(house_98105\$SqFtTotLiving, p=c(.25, .5, .75))
lm_spline <- lm(AdjSalePrice ~ bs(SqFtTotLiving, knots=knots, degree=3) +
                SqFtLot + Bathrooms + Bedrooms + BldgGrade, data=house_98105)

# **32. Generalized Additive Models (GAM)**
### GAM mengotomatiskan penentuan lokasi dan jumlah *knots* pada pemodelan *spline* non-linier tanpa harus ditentukan manual oleh pengguna.

**Code Reproduction: R:**

In [ ]:
library(mgcv)
lm_gam <- gam(AdjSalePrice ~ s(SqFtTotLiving) + SqFtLot + Bathrooms + Bedrooms + BldgGrade,
              data=house_98105)